# Library OOP Practical Assignment 1

**NAME:** AJAK DENG GARANG DENG  

**REG_NO:** M25B38/031  

**COURSE:** BSDS  

**YEAR:** 2:1  


## QN.1  Identify classes, attributes and methods

**(i) Student class  # A Student represents a student who can borrow books.**

**Attributes**

name     - student's name

reg_no   - registration number

contact  - student's contact

loans    - loans currently associated with the student



**Methods:**

__init__()   - creates a student object

__str__()    - gives a readable description of the student to the users.

add_loan()

list_loans()

**(ii) Book class  # A Book represents a particular book/title in the library.**

**Attributes:**

title       - title of the book

Author      - author of the book

copies      - copies belonging to that book

catalogue   - stores all registered book titles

isbn        - checks whether an ISBN has 10 or 13 digits


**Methods:**

__init__()      - creates a book

all_titles()    - returns all book titles

is_valid_isbn() - checks whether an ISBN has 10 or 13 digits

__str__()

**(iii) BookCopy class  # A BookCopy represents one physical copy of a book.**

**Attributes:**

copy_id   - unique ID of the physical copy

book      - the book to which the copy belongs

is_available - tells whether the copy is available

**Methods:**

__init__()

__str__()

__repr__()

mark_borrowed() - tells the book borrowed

mark_returned() - tells the book returned

**(iv) Loan class   # A Loan represents the borrowing of a book copy by a student.**

**Attributes:**

student

copy

borrow_date

due_date

return_date

**It also has the class attribute:**

FINE_PER_DAY = 1000


**Methods:**

__init__()

calculate_fine()'

return_book()

__str__()

## QN.2 – Implement Student, Book, BookCopy and Loan

In [7]:
from datetime import date

FINE_PER_DAY = 1000 # The library charges UGX 1,000 for every day a book is late.


class Student:   # A Student stores personal details and their loans.
    def __init__(self, name, reg_no, contact):
        self.name = name
        self.reg_no = reg_no
        self.contact = contact
        self.loans = []  # Stores books borrowed by this student.

    def add_loan(self, loan): # Add a new loan to the student's loan list.
        self.loans.append(loan)  # we save each loan so we can later list all loans for this student.

    def list_loans(self):   # Return all loans belonging to this student.
        return self.loans

    def __str__(self):
        # Give the student a simple readable description.
        return f"{self.reg_no} - {self.name}"


class Book: 

    catalogue = [] # This list stores every book registered in the library.

    def __init__(self, isbn, title, author):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.copies = []  # Physical copies belonging to this book.

        Book.catalogue.append(self) # Add the new book to the library catalogue.

    @classmethod
    def all_titles(cls):
        return [book.title for book in cls.catalogue] # Return the titles of all registered books.

    @classmethod
    def is_valid_isbn(cls, isbn):
        clean_isbn = isbn.replace("-", "") # Remove hyphens and check for 10 or 13 digits.
        return len(clean_isbn) in (10, 13)

    def __str__(self):
        return f"{self.title} by {self.author} (ISBN: {self.isbn})"


class BookCopy:
    # A BookCopy represents one physical copy of a book.
    def __init__(self, copy_id, book):
        self.copy_id = copy_id
        self.book = book
        self.is_available = True

        book.copies.append(self) # Keep this physical copy under its book.

    def __str__(self):   # __str__ readable for users
        status = "available" if self.is_available else "borrowed"
        return f"{self.copy_id}: {self.book.title} ({status})"

    def __repr__(self): # __repr__ unamniguous for developers
        return f"BookCopy({self.copy_id!r}, {self.book.title!r})"

    def mark_borrowed(self):
        if not self.is_available: # A copy becomes unavailable when it is borrowed.
            raise ValueError("This book copy is already borrowed.")

        self.is_available = False

    def mark_returned(self):
        self.is_available = True  # Make the copy available after it is returned.


class Loan:  # A Loan connects one student with one physical book copy.
    def __init__(self, student, copy, borrow_date, due_date):
        self.student = student
        self.copy = copy
        self.borrow_date = borrow_date
        self.due_date = due_date
        self.return_date = None

    def return_book(self, return_date):   # Store the return date and make the copy available.
        self.return_date = return_date
        self.copy.mark_returned()

    def calculate_fine(self):   # No fine is charged until the book is returned.
        if self.return_date is None:
            return 0

        late_days = (self.return_date - self.due_date).days # Find how many days after the due date the book was returned.  # If late_days is negative, the book was returned early.Early or on-time returns have no fine.
        late_days = max(0, late_days)

        return late_days * FINE_PER_DAY

    def __str__(self):
        if self.return_date is None:
            status = "Not returned"
        else:
            status = f"Returned: {self.return_date}"

        return (
            f"Loan: {self.student.reg_no} - {self.student.name} -> "
            f"{self.copy.book.title} "
            f"(Due: {self.due_date}, {status})"
        )


## QN.3 Implement `mark_borrowed`, `mark_returned` and a simple borrow flow

The borrowing flow is:
 **Student chooses a copy → check availability → mark the copy borrowed → create a Loan → add the loan to the student.**

A returned book follows the reverse process: the loan records the return date and the copy is marked available.

In [8]:
def borrow_book(student, copy, borrow_date, due_date):
    
    if not copy.is_available: # First check whether this physical copy is available.
        print(f"Borrowing failed: copy {copy.copy_id} "
            f"is already borrowed.")
        return None

    copy.mark_borrowed() # Mark the physical copy as borrowed.

    loan = Loan(student, copy, borrow_date, due_date) # Create a loan for the student.

    student.add_loan(loan)  # Save the loan under the student.

    print(f"Borrowing successful: {student.name} "
        f"borrowed '{copy.book.title}'.")
    return loan


def return_book(loan, return_date):
   
    loan.return_book(return_date)  # Record the return date.

    fine = loan.calculate_fine() # Calculate the fine after the return.

    print(f"Return successful: '{loan.copy.book.title}' "
        f"was returned by {loan.student.name}.")
    print(f"Fine: UGX {fine:,}")

    return fine


## QN.4  Implement `calculate_fine()` using `FINE_PER_DAY = 1000`

The calculation is:  **Fine = number of late days × 1,000**

For example, a book due on **7 September 2026** and returned on **10 September 2026** is 3 days late, so the fine is **UGX 3,000**.

**Fine = 3 × 1,000 = UGX 3,000**

If the book is returned on time or early, the fine is **UGX 0**.

In [13]:

# Book information used in the test.
book1 = Book("978-0000001","Python Energy", "Ajak D. Garang"
)

# Physical copy of Python Energy.
copy1 = BookCopy("CP001", book1)

# Student information.
student1 = Student("Deng Guttier", "M25B38/031","+256-762-075-084")

# Deng Guttier borrows the book on 1 September. and the book is due on 7 September.
loan1 = borrow_book(student1, copy1, date(2026, 9, 1), date(2026, 9, 7))

# The book is returned on 10 September. And the book is 3 days late.
fine1 = return_book(loan1,date(2026, 9, 10))

print("Expected fine: UGX 3,000")
print("Calculated fine: UGX", fine1)


Borrowing successful: Deng Guttier borrowed 'Python Energy'.
Return successful: 'Python Energy' was returned by Deng Guttier.
Fine: UGX 3,000
Expected fine: UGX 3,000
Calculated fine: UGX 3000


## QN.5 – Small demo script

The following demo:
1. registers students,
2. adds book copies,
3. borrows a book,
4. returns a book,
5. prints the final status.

In [14]:

# Register two students.
student1 = Student("Deng Guttier","M25B38/031","+256-762-075-084")

student2 = Student("Amou G. Atem","M25B38/032","+256-763-934-904")

print("REGISTERED STUDENTS")
print(student1)
print(student2)

# Register two books.
book1 = Book("978-0000001","Python Energy","Ajak D. Garang")

book2 = Book("978-0000002","Master Data Structures","Mr. Geno")

# Create physical copies.
copy1 = BookCopy("CP001", book1)
copy2 = BookCopy("CP002", book2)

print("\nBOOK COPIES")
print(copy1)
print(copy2)

# Garang borrows Python Basics on 10 September.
# It is due on 17 September.
loan1 = borrow_book(student1, copy1, date(2026, 9, 10), date(2026, 9, 17))

print("\nSTATUS AFTER BORROWING")
print(copy1)

# Deng Guttier returns the book two days late on 19 September.
return_book(loan1, date(2026, 9, 19))

print("\nSTATUS AFTER RETURN")
print(copy1)

print("\nLOAN DETAILS")
print(loan1)


REGISTERED STUDENTS
M25B38/031 - Deng Guttier
M25B38/032 - Amou G. Atem

BOOK COPIES
CP001: Python Energy (available)
CP002: Master Data Structures (available)
Borrowing successful: Deng Guttier borrowed 'Python Energy'.

STATUS AFTER BORROWING
CP001: Python Energy (borrowed)
Return successful: 'Python Energy' was returned by Deng Guttier.
Fine: UGX 2,000

STATUS AFTER RETURN
CP001: Python Energy (available)

LOAN DETAILS
Loan: M25B38/031 - Deng Guttier -> Python Energy (Due: 2026-09-17, Returned: 2026-09-19)


## QN.6 – Challenge

### A. Prevent borrowing an unavailable copy

The borrow_book() function already checks copy.available.

If the copy is unavailable, the function stops and returns None. This prevents two students from borrowing the same physical copy at the same time.


In [15]:
# Peter is already registered above. Create a third physical copy of Python Basics.
copy3 = BookCopy("CP003", book1)

# Deng Guttier borrows CP003 first.
loan_a = borrow_book(student1, copy3, date(2026, 9, 20), date(2026, 9, 25)
)

# Peter tries to borrow the same physical copy. This should fail because CP003 is already borrowed.
loan_b = borrow_book( student2, copy3, date(2026, 9, 21), date(2026, 9, 26))

print("\nCURRENT COPY STATUS")
print(copy3)

# Deng Guttier returns CP003 on the due date.
return_book( loan_a, date(2026, 9, 25))

print("\nSTATUS AFTER RETURNING CP003")
print(copy3)


Borrowing successful: Deng Guttier borrowed 'Python Energy'.
Borrowing failed: copy CP003 is already borrowed.

CURRENT COPY STATUS
CP003: Python Energy (borrowed)
Return successful: 'Python Energy' was returned by Deng Guttier.
Fine: UGX 0

STATUS AFTER RETURNING CP003
CP003: Python Energy (available)


### B. List all loans for a student

The Student class stores every loan in self.loans. The list_loans() method returns that list.

In [16]:

# Create another physical copy of Data Structures.
copy4 = BookCopy("CP004", book2)

# Deng Guttier borrows Data Structures.
loan_c = borrow_book(student1, copy4, date(2026, 9, 22), date(2026, 9, 28))

print(f"\nALL LOANS FOR {student1.name}")

# Display every loan stored for Garang.
for number, loan in enumerate(student1.list_loans(), start=1):
    print(f"{number}. {loan}")


Borrowing successful: Deng Guttier borrowed 'Master Data Structures'.

ALL LOANS FOR Deng Guttier
1. Loan: M25B38/031 - Deng Guttier -> Python Energy (Due: 2026-09-17, Returned: 2026-09-19)
2. Loan: M25B38/031 - Deng Guttier -> Python Energy (Due: 2026-09-25, Returned: 2026-09-25)
3. Loan: M25B38/031 - Deng Guttier -> Master Data Structures (Due: 2026-09-28, Not returned)
